# Unit07 區域性離群因子 (Local Outlier Factor, LOF) | 批次反應器多模式異常檢測

本 Notebook 以程式模擬的批次反應器數據（三種配方的溫度、壓力、流量，另含 30 筆異常批次）建立 LOF 異常檢測模型，識別多模式操作下的異常批次。數據由程式直接生成，不需要外部資料檔。

## 目標
- 理解 LOF 演算法的局部密度概念
- 學會使用 scikit-learn 實作 LOF 模型（`novelty=False` 與 `novelty=True` 兩種模式）
- 掌握 LOF 超參數 `n_neighbors` 與 `contamination` 的作用
- 應用 LOF 於化工製程多模式異常檢測，並以 AUC、Precision、Recall 評估
- 以多組隨機種子與多密度範例，客觀比較 LOF 與 Isolation Forest

資料背景：模擬批次反應器在三種配方下的操作數據（溫度、壓力、流量），包含少量異常操作案例。

---
## 0. 環境設定

In [1]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P2_Unit07_LOF'
SOURCE_DATA_DIR = 'batch_reactor'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3590'
    os.remove(shortcut_path)
except FileNotFoundError:
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Part_2' / 'Unit07'
        OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        DATA_DIR = NOTEBOOK_DIR / 'data' / SOURCE_DATA_DIR
        FIG_DIR = OUTPUT_DIR / 'figs'
        MODEL_DIR = OUTPUT_DIR / 'models'
    else:
        print(f"⚠️ 找不到雲端 ChemE-3590 路徑，請確認資料夾是否正確")
else:
  NOTEBOOK_DIR = Path.cwd()
  OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
  DATA_DIR = NOTEBOOK_DIR / 'data' / SOURCE_DATA_DIR
  MODEL_DIR = OUTPUT_DIR / 'models'
  FIG_DIR = OUTPUT_DIR / 'figs'

# 建立必要資料夾
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 數據來源目錄: {DATA_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: D:\MyGit\ChemE-3590\Part_2\Unit07
✓ 數據來源目錄: D:\MyGit\ChemE-3590\Part_2\Unit07\data\batch_reactor
✓ 結果輸出目錄: D:\MyGit\ChemE-3590\Part_2\Unit07\outputs\P2_Unit07_LOF
✓ 模型輸出目錄: D:\MyGit\ChemE-3590\Part_2\Unit07\outputs\P2_Unit07_LOF\models
✓ 圖檔輸出目錄: D:\MyGit\ChemE-3590\Part_2\Unit07\outputs\P2_Unit07_LOF\figs


In [2]:
# ========================================
# 載入相關套件
# ========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# 設定隨機種子以確保結果可重現
SEED = 42
np.random.seed(SEED)

# 圖中文字一律使用英文，避免 Colab 缺少中文字型
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

print("✓ 套件載入完成")

✓ 套件載入完成


---
## 1. 數據生成與準備

### 1.1 模擬批次反應器數據（優化版）

模擬三種配方的批次反應器操作數據，並增加數據複雜度以提升模型評估的真實性：

**正常操作數據（330 批次，91.7%）**：
- **配方 A**：高溫高壓反應（150 批次，增加變異性）
- **配方 B**：中溫中壓反應（100 批次，增加變異性）
- **配方 C**：低溫低壓反應（80 批次，增加變異性）

**異常操作數據（30 批次，8.3%）**：
採用分層異常策略，模擬真實工業場景中不同嚴重程度的異常：
- **輕微異常（12 批次，40%）**：邊界案例，接近正常範圍，難以檢測
  - 接近配方 A/B/C 但微偏離
  - 介於兩配方之間的過渡區
- **中度異常（12 批次，40%）**：明顯偏離但不極端
  - 溫度/壓力/流量單一參數偏離
  - 參數間不匹配（如高溫低壓）
- **嚴重異常（6 批次，20%）**：極端偏離，容易檢測
  - 極端高溫高壓或全面偏低
  - 嚴重的參數不匹配

**挑戰性提升**：
- 增加正常數據的標準差（模擬操作波動）
- 輕微異常接近正常範圍（增加假陰性風險）
- 正常與異常數據有部分重疊（降低 AUC，更真實）

In [3]:
# ========================================
# 1.1 模擬批次反應器數據（三種配方 + 分層異常）
# ========================================
feature_names = ['Temperature', 'Pressure', 'Flow_Rate']


def generate_batch_reactor_data(seed=42):
    # ========================================
    # 1.1 模擬批次反應器數據（優化版 - 增加挑戰性）
    # ========================================
    np.random.seed(seed)

    # 增加正常數據的樣本量與變異性
    # 配方 A：高溫高壓（150 批次）- 增加標準差
    recipe_A = np.random.normal(loc=[85, 5.0, 2.5], scale=[4, 0.4, 0.25], size=(150, 3))

    # 配方 B：中溫中壓（100 批次）- 增加標準差
    recipe_B = np.random.normal(loc=[65, 3.5, 1.8], scale=[4, 0.4, 0.2], size=(100, 3))

    # 配方 C：低溫低壓（80 批次）- 增加標準差
    recipe_C = np.random.normal(loc=[45, 2.5, 1.2], scale=[3, 0.3, 0.15], size=(80, 3))

    # 正常數據
    X_normal = np.vstack([recipe_A, recipe_B, recipe_C])
    n_normal = len(X_normal)  # 330 個正常樣本

    # 生成分層異常數據（輕微 + 中度 + 嚴重）
    n_anomaly = 30  # 總異常數約 8.3%
    n_mild = 12      # 40% 輕微異常（難檢測）
    n_moderate = 12  # 40% 中度異常
    n_severe = 6     # 20% 嚴重異常

    anomaly_list = []

    # 1. 輕微異常（邊界案例 - 接近正常範圍）
    for i in range(n_mild):
        anomaly_type = np.random.choice(['A_boundary', 'B_boundary', 'C_boundary', 'transition'])
    
        if anomaly_type == 'A_boundary':
            # 接近配方 A 但微偏離
            sample = np.random.normal([90, 5.4, 2.7], [2, 0.2, 0.1])
        elif anomaly_type == 'B_boundary':
            # 接近配方 B 但微偏離
            sample = np.random.normal([70, 3.8, 2.0], [2, 0.2, 0.1])
        elif anomaly_type == 'C_boundary':
            # 接近配方 C 但微偏離
            sample = np.random.normal([48, 2.8, 1.4], [1.5, 0.15, 0.08])
        else:
            # 介於兩配方之間的過渡區
            sample = np.random.normal([75, 4.2, 2.2], [2, 0.2, 0.1])
    
        anomaly_list.append(sample)

    # 2. 中度異常（明顯偏離但不極端）
    for i in range(n_moderate):
        anomaly_type = np.random.choice(['temp_high', 'pressure_high', 'flow_low', 'mismatch'])
    
        if anomaly_type == 'temp_high':
            # 溫度明顯偏高
            base_recipe = np.random.choice(['A', 'B', 'C'])
            if base_recipe == 'A':
                sample = np.random.normal([95, 5.2, 2.6], [2, 0.2, 0.1])
            elif base_recipe == 'B':
                sample = np.random.normal([75, 3.7, 1.9], [2, 0.2, 0.1])
            else:
                sample = np.random.normal([52, 2.6, 1.3], [1.5, 0.15, 0.08])
    
        elif anomaly_type == 'pressure_high':
            # 壓力明顯偏高
            base_recipe = np.random.choice(['A', 'B', 'C'])
            if base_recipe == 'A':
                sample = np.random.normal([87, 5.8, 2.5], [2, 0.2, 0.1])
            elif base_recipe == 'B':
                sample = np.random.normal([67, 4.2, 1.8], [2, 0.2, 0.1])
            else:
                sample = np.random.normal([47, 3.2, 1.2], [1.5, 0.15, 0.08])
    
        elif anomaly_type == 'flow_low':
            # 流量明顯偏低
            base_recipe = np.random.choice(['A', 'B', 'C'])
            if base_recipe == 'A':
                sample = np.random.normal([85, 5.0, 1.8], [2, 0.2, 0.1])
            elif base_recipe == 'B':
                sample = np.random.normal([65, 3.5, 1.2], [2, 0.2, 0.1])
            else:
                sample = np.random.normal([45, 2.5, 0.8], [1.5, 0.15, 0.08])
    
        else:
            # 參數不匹配（溫度與壓力不協調）
            sample = np.random.normal([78, 4.8, 1.5], [3, 0.3, 0.15])
    
        anomaly_list.append(sample)

    # 3. 嚴重異常（極端偏離 - 容易檢測）
    severe_anomalies = [
        [100, 6.2, 3.2],   # 配方 A 極端：溫度壓力過高
        [55, 2.0, 0.5],    # 配方 C 極端：全面偏低
        [95, 3.0, 1.0],    # 嚴重不匹配：高溫低壓
        [60, 5.5, 0.8],    # 嚴重不匹配：中溫高壓低流量
        [40, 4.5, 2.5],    # 配方 C 異常：低溫高壓高流量
        [88, 2.5, 3.0],    # 極端不匹配：高溫低壓高流量
    ]

    for sample in severe_anomalies:
        anomaly_list.append(np.array(sample))

    # 轉換為 numpy array
    X_anomaly = np.array(anomaly_list)

    # 合併數據
    X = np.vstack([X_normal, X_anomaly])

    # 建立 DataFrame
    df = pd.DataFrame(X, columns=feature_names)
    df['Recipe'] = ['A']*150 + ['B']*100 + ['C']*80 + ['Anomaly']*n_anomaly
    df['True_Label'] = [1]*n_normal + [-1]*n_anomaly  # 1: 正常, -1: 異常
    df['Anomaly_Severity'] = ['Normal']*n_normal + \
                             ['Mild']*n_mild + \
                             ['Moderate']*n_moderate + \
                             ['Severe']*n_severe
    return df


df = generate_batch_reactor_data(seed=SEED)
n_normal = int((df['True_Label'] == 1).sum())
n_anomaly = int((df['True_Label'] == -1).sum())

print("="*60)
print("數據生成完成")
print("="*60)
print(f"總批次數: {len(df)}")
print(f"正常批次: {np.sum(df['True_Label'] == 1)} ({100*np.sum(df['True_Label'] == 1)/len(df):.1f}%)")
print(f"異常批次: {np.sum(df['True_Label'] == -1)} ({100*np.sum(df['True_Label'] == -1)/len(df):.1f}%)")
print(f"\n各配方批次數:")
print(df['Recipe'].value_counts())
print(f"\n異常嚴重程度分布:")
print(df['Anomaly_Severity'].value_counts())
print(f"\n【數據設計說明】")
print("  • 40% 為輕微異常（邊界案例），與正常數據部分重疊")
print("  • 正常數據含操作波動，正常與異常的 LOF 分數分布有重疊區")
print("  • 完美分離（F1 = 1.0、AUC = 1.0）在真實製程數據中並不常見")
print(f"\n數據概覽:")
df.head(10)

數據生成完成
總批次數: 360
正常批次: 330 (91.7%)
異常批次: 30 (8.3%)

各配方批次數:
Recipe
A          150
B          100
C           80
Anomaly     30
Name: count, dtype: int64

異常嚴重程度分布:
Anomaly_Severity
Normal      330
Mild         12
Moderate     12
Severe        6
Name: count, dtype: int64

【數據設計說明】
  • 40% 為輕微異常（邊界案例），與正常數據部分重疊
  • 正常數據含操作波動，正常與異常的 LOF 分數分布有重疊區
  • 完美分離（F1 = 1.0、AUC = 1.0）在真實製程數據中並不常見

數據概覽:


,Temperature,Pressure,Flow_Rate,Recipe,True_Label,Anomaly_Severity
0,86.986857,4.944694,2.661922,A,1,Normal
1,91.092119,4.906339,2.441466,A,1,Normal
2,91.316851,5.306974,2.382631,A,1,Normal
3,87.170240,4.814633,2.383568,A,1,Normal
4,85.967849,4.234688,2.068771,A,1,Normal
5,82.750850,4.594868,2.578562,A,1,Normal
6,81.367904,4.435079,2.866412,A,1,Normal
7,84.096895,5.027011,2.143813,A,1,Normal
8,82.822469,5.044369,2.212252,A,1,Normal
9,86.502792,4.759745,2.427077,A,1,Normal


### 1.2 數據統計描述

In [4]:
# ========================================
# 1.2 各配方數據統計描述
# ========================================
print("各配方操作參數統計：")
print("="*60)
for recipe in ['A', 'B', 'C']:
    print(f"\n配方 {recipe}:")
    print(df[df['Recipe'] == recipe][feature_names].describe())

各配方操作參數統計：

配方 A:
       Temperature    Pressure   Flow_Rate
count   150.000000  150.000000  150.000000
mean     85.289553    4.950803    2.523622
std       3.521603    0.391869    0.260304
min      77.191649    3.703493    1.845064
25%      82.746539    4.673704    2.358971
50%      85.471941    4.946319    2.524334
75%      87.147027    5.272546    2.684743
max      93.759212    5.925863    3.463183

配方 B:
       Temperature    Pressure   Flow_Rate
count   100.000000  100.000000  100.000000
mean     65.159259    3.433257    1.785413
std       3.688166    0.416745    0.206793
min      55.792315    2.421245    1.269806
25%      62.541472    3.136488    1.641224
50%      64.988587    3.360121    1.789012
75%      67.433965    3.700754    1.913273
max      75.293439    4.731552    2.289150

配方 C:
       Temperature   Pressure  Flow_Rate
count    80.000000  80.000000  80.000000
mean     45.081307   2.564094   1.228299
std       2.841416   0.286619   0.144446
min      38.403582   2.028933 

### 1.3 數據視覺化探索

In [5]:
# ========================================
# 1.3 原始數據視覺化
# ========================================
fig = plt.figure(figsize=(16, 10))

# 定義顏色
colors = {'A': 'blue', 'B': 'green', 'C': 'orange', 'Anomaly': 'red'}

# 3D 散點圖
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
for recipe in ['A', 'B', 'C', 'Anomaly']:
    mask = df['Recipe'] == recipe
    ax1.scatter(df.loc[mask, 'Temperature'], 
                df.loc[mask, 'Pressure'],
                df.loc[mask, 'Flow_Rate'],
                c=colors[recipe], label=('Anomaly' if recipe == 'Anomaly' else f'Recipe {recipe}'), 
                alpha=0.7 if recipe != 'Anomaly' else 1.0,
                s=30 if recipe != 'Anomaly' else 150,
                marker='o' if recipe != 'Anomaly' else 'x')

ax1.set_xlabel('Temperature (C)')
ax1.set_ylabel('Pressure (bar)')
ax1.set_zlabel('Flow Rate (m3/h)')
ax1.set_title('3D Process Data Visualization')
ax1.legend()

# Temperature vs Pressure
ax2 = fig.add_subplot(2, 2, 2)
for recipe in ['A', 'B', 'C', 'Anomaly']:
    mask = df['Recipe'] == recipe
    ax2.scatter(df.loc[mask, 'Temperature'], 
                df.loc[mask, 'Pressure'],
                c=colors[recipe], label=('Anomaly' if recipe == 'Anomaly' else f'Recipe {recipe}'),
                alpha=0.7 if recipe != 'Anomaly' else 1.0,
                s=50 if recipe != 'Anomaly' else 150,
                marker='o' if recipe != 'Anomaly' else 'x')

ax2.set_xlabel('Temperature (C)')
ax2.set_ylabel('Pressure (bar)')
ax2.set_title('Temperature vs Pressure')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Temperature vs Flow Rate
ax3 = fig.add_subplot(2, 2, 3)
for recipe in ['A', 'B', 'C', 'Anomaly']:
    mask = df['Recipe'] == recipe
    ax3.scatter(df.loc[mask, 'Temperature'], 
                df.loc[mask, 'Flow_Rate'],
                c=colors[recipe], label=('Anomaly' if recipe == 'Anomaly' else f'Recipe {recipe}'),
                alpha=0.7 if recipe != 'Anomaly' else 1.0,
                s=50 if recipe != 'Anomaly' else 150,
                marker='o' if recipe != 'Anomaly' else 'x')

ax3.set_xlabel('Temperature (C)')
ax3.set_ylabel('Flow Rate (m3/h)')
ax3.set_title('Temperature vs Flow Rate')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Pressure vs Flow Rate
ax4 = fig.add_subplot(2, 2, 4)
for recipe in ['A', 'B', 'C', 'Anomaly']:
    mask = df['Recipe'] == recipe
    ax4.scatter(df.loc[mask, 'Pressure'], 
                df.loc[mask, 'Flow_Rate'],
                c=colors[recipe], label=('Anomaly' if recipe == 'Anomaly' else f'Recipe {recipe}'),
                alpha=0.7 if recipe != 'Anomaly' else 1.0,
                s=50 if recipe != 'Anomaly' else 150,
                marker='o' if recipe != 'Anomaly' else 'x')

ax4.set_xlabel('Pressure (bar)')
ax4.set_ylabel('Flow Rate (m3/h)')
ax4.set_title('Pressure vs Flow Rate')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(wspace=0.4)
plt.savefig(FIG_DIR / 'data_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 數據視覺化完成")

✓ 數據視覺化完成


---
## 2. 數據預處理

### 2.1 數據標準化

In [6]:
# ========================================
# 2.1 數據標準化
# ========================================
# 提取特徵數據
X = df[feature_names].values
y_true = df['True_Label'].values

# 標準化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("="*60)
print("數據標準化完成")
print("="*60)
print(f"原始數據形狀: {X.shape}")
print(f"標準化後數據形狀: {X_scaled.shape}")
print(f"\n標準化前數據範圍:")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X[:, i].min():.2f}, {X[:, i].max():.2f}]")
print(f"\n標準化後數據範圍:")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X_scaled[:, i].min():.2f}, {X_scaled[:, i].max():.2f}]")

數據標準化完成
原始數據形狀: (360, 3)
標準化後數據形狀: (360, 3)

標準化前數據範圍:
  Temperature: [38.40, 100.00]
  Pressure: [2.00, 6.20]
  Flow_Rate: [0.50, 3.46]

標準化後數據範圍:
  Temperature: [-1.89, 1.83]
  Pressure: [-1.81, 2.13]
  Flow_Rate: [-2.53, 2.56]


---
## 3. LOF 模型建立與訓練

### 3.1 建立基礎 LOF 模型

In [7]:
# ========================================
# 3.1 建立並訓練 LOF 模型
# ========================================
lof = LocalOutlierFactor(
    n_neighbors=20,      # 鄰近點數量
    contamination=0.083,  # 教學用：假設已知異常比例 30/360
    novelty=False,       # 內部異常檢測模式
    metric='euclidean'   # 歐氏距離
)

# 訓練並預測
y_pred_lof = lof.fit_predict(X_scaled)
lof_scores = -lof.negative_outlier_factor_  # 轉為正值，越大越異常
lof_threshold = -lof.offset_                # contamination 換算出的 LOF 判定門檻

# 將結果加入 DataFrame
df['LOF_Prediction'] = ['Normal' if p == 1 else 'Anomaly' for p in y_pred_lof]
df['LOF_Score'] = lof_scores

print("="*60)
print("LOF 模型訓練完成")
print("="*60)
print(f"總批次數: {len(df)}")
print(f"檢測到的異常批次: {np.sum(y_pred_lof == -1)}")
print(f"LOF 分數範圍: {lof_scores.min():.3f} ~ {lof_scores.max():.3f}")
print(f"contamination=0.083 對應的判定門檻: LOF > {lof_threshold:.3f}")
print(f"\n各配方的異常檢測結果:")
print(df.groupby('Recipe')['LOF_Prediction'].value_counts())

LOF 模型訓練完成
總批次數: 360
檢測到的異常批次: 30
LOF 分數範圍: 0.940 ~ 4.282
contamination=0.083 對應的判定門檻: LOF > 1.421

各配方的異常檢測結果:
Recipe   LOF_Prediction
A        Normal            142
         Anomaly             8
Anomaly  Anomaly            15
         Normal             15
B        Normal             95
         Anomaly             5
C        Normal             78
         Anomaly             2
Name: count, dtype: int64


### 3.2 超參數調整：測試不同的 n_neighbors

以 AUC（與閾值無關）和 F1 同時評估 $k$ 的影響。

> 注意 1：`contamination` 固定為真實異常比例，所以預測異常數恆為 30，此時 Precision = Recall = F1，不需分開繪製。
>
> 注意 2：此處用真實標籤選 $k$ 僅為教學示範。實務上應使用獨立驗證集，或檢查鄰近 $k$ 值的結果是否穩定。

In [8]:
# ========================================
# 3.2 測試不同 n_neighbors 值的影響
# ========================================
k_values = [5, 10, 15, 20, 25, 30, 40, 50]
y_is_anomaly = (y_true == -1)
results = []

for k in k_values:
    lof_temp = LocalOutlierFactor(n_neighbors=k, contamination=0.083)
    y_pred_temp = lof_temp.fit_predict(X_scaled)
    lof_scores_temp = -lof_temp.negative_outlier_factor_

    results.append({
        'k': k,
        'n_anomalies': int(np.sum(y_pred_temp == -1)),
        'avg_lof_score': lof_scores_temp.mean(),
        'auc': roc_auc_score(y_is_anomaly, lof_scores_temp),
        'f1_score': f1_score(y_true, y_pred_temp, pos_label=-1, zero_division=0),
    })
    r = results[-1]
    print(f"k={k:2d} | 異常數: {r['n_anomalies']:2d} | 平均LOF: {r['avg_lof_score']:.3f} | "
          f"AUC: {r['auc']:.3f} | F1(=P=R): {r['f1_score']:.3f}")

results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(results_df['k'], results_df['auc'], marker='o', linewidth=2, color='tab:blue')
axes[0].set_xlabel('n_neighbors (k)')
axes[0].set_ylabel('AUC')
axes[0].set_title('AUC vs k (threshold-free)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(results_df['k'], results_df['f1_score'], marker='o', linewidth=2, color='tab:green')
axes[1].set_xlabel('n_neighbors (k)')
axes[1].set_ylabel('F1-Score (= Precision = Recall)')
axes[1].set_title('F1-Score vs k (30 flagged)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(results_df['k'], results_df['avg_lof_score'], marker='o', linewidth=2, color='tab:orange')
axes[2].set_xlabel('n_neighbors (k)')
axes[2].set_ylabel('Average LOF Score')
axes[2].set_title('Average LOF Score vs k')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'hyperparameter_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

best_k_auc = int(results_df.loc[results_df['auc'].idxmax(), 'k'])
best_k_f1 = int(results_df.loc[results_df['f1_score'].idxmax(), 'k'])
print(f"\nAUC 最高的 k: {best_k_auc}；F1 最高的 k: {best_k_f1}")
print("本 Notebook 後續統一使用 k = 20（預設值，落在 F1 與 AUC 皆穩定的區間）")

k= 5 | 異常數: 30 | 平均LOF: 1.165 | AUC: 0.715 | F1(=P=R): 0.400
k=10 | 異常數: 30 | 平均LOF: 1.157 | AUC: 0.801 | F1(=P=R): 0.433
k=15 | 異常數: 30 | 平均LOF: 1.158 | AUC: 0.828 | F1(=P=R): 0.500
k=20 | 異常數: 30 | 平均LOF: 1.155 | AUC: 0.835 | F1(=P=R): 0.500
k=25 | 異常數: 30 | 平均LOF: 1.150 | AUC: 0.842 | F1(=P=R): 0.467
k=30 | 異常數: 30 | 平均LOF: 1.144 | AUC: 0.844 | F1(=P=R): 0.467
k=40 | 異常數: 30 | 平均LOF: 1.129 | AUC: 0.849 | F1(=P=R): 0.433
k=50 | 異常數: 30 | 平均LOF: 1.111 | AUC: 0.850 | F1(=P=R): 0.433



AUC 最高的 k: 50；F1 最高的 k: 15
本 Notebook 後續統一使用 k = 20（預設值，落在 F1 與 AUC 皆穩定的區間）


---
## 4. 模型評估與結果分析

### 4.1 模型效能評估

In [9]:
# ========================================
# 4.1 模型效能評估
# ========================================
# 混淆矩陣
cm = confusion_matrix(y_true, y_pred_lof, labels=[1, -1])
precision = precision_score(y_true, y_pred_lof, pos_label=-1)
recall = recall_score(y_true, y_pred_lof, pos_label=-1)
f1 = f1_score(y_true, y_pred_lof, pos_label=-1)

print("="*60)
print("LOF 模型效能評估")
print("="*60)
print("\n混淆矩陣:")
print(f"{'':>15} {'Predicted Normal':>20} {'Predicted Anomaly':>20}")
print(f"{'True Normal':>15} {cm[0,0]:>20} {cm[0,1]:>20}")
print(f"{'True Anomaly':>15} {cm[1,0]:>20} {cm[1,1]:>20}")

print(f"\n評估指標:")
print(f"  Precision (精確率): {precision:.3f}")
print(f"  Recall (召回率):    {recall:.3f}")
print(f"  F1-Score:           {f1:.3f}")

print("\n分類報告:")
print(classification_report(y_true, y_pred_lof, 
                            labels=[1, -1],
                            target_names=['Normal', 'Anomaly']))

# 視覺化混淆矩陣
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：混淆矩陣熱力圖（數值）
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'],
            cbar_kws={'label': 'Count'},
            linewidths=2, linecolor='black')
ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax1.set_title('Confusion Matrix (Count)', fontsize=14, fontweight='bold')

# 右圖：混淆矩陣熱力圖（百分比）
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
ax2 = axes[1]
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Greens', ax=ax2,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'],
            cbar_kws={'label': 'Percentage (%)'},
            linewidths=2, linecolor='black')
ax2.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax2.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax2.set_title('Confusion Matrix (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ 混淆矩陣視覺化完成")

LOF 模型效能評估

混淆矩陣:
                    Predicted Normal    Predicted Anomaly
    True Normal                  315                   15
   True Anomaly                   15                   15

評估指標:
  Precision (精確率): 0.500
  Recall (召回率):    0.500
  F1-Score:           0.500

分類報告:
              precision    recall  f1-score   support

      Normal       0.95      0.95      0.95       330
     Anomaly       0.50      0.50      0.50        30

    accuracy                           0.92       360
   macro avg       0.73      0.73      0.73       360
weighted avg       0.92      0.92      0.92       360




✓ 混淆矩陣視覺化完成


### 4.2 ROC 曲線與閾值分析

In [10]:
# ========================================
# 4.2 ROC 曲線與閾值分析
# ========================================
from sklearn.metrics import roc_curve, auc

# 計算 ROC 曲線
# 將標籤轉換為二元格式：1=異常（正類），0=正常（負類）
y_true_binary = (y_true == -1).astype(int)
# LOF 分數越高越異常，直接使用
fpr, tpr, thresholds = roc_curve(y_true_binary, lof_scores)
roc_auc = auc(fpr, tpr)

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：ROC 曲線
ax1 = axes[0]
ax1.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'LOF (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
         label='Random Classifier')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve for LOF')
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.3)

# 右圖：LOF 分數分布與閾值
ax2 = axes[1]
normal_scores = df[df['True_Label'] == 1]['LOF_Score'].values
anomaly_scores = df[df['True_Label'] == -1]['LOF_Score'].values

score_bins = np.linspace(lof_scores.min(), lof_scores.max(), 40)
ax2.hist(normal_scores, bins=score_bins, alpha=0.6, label='Normal', color='blue', density=True)
ax2.hist(anomaly_scores, bins=score_bins, alpha=0.6, label='Anomaly', color='red', density=True)

# 標示建議閾值
ax2.axvline(x=1.5, color='orange', linestyle='--', linewidth=2, 
            label='Warning Threshold (1.5)')
ax2.axvline(x=2.0, color='darkred', linestyle='--', linewidth=2, 
            label='Critical Threshold (2.0)')

ax2.set_xlabel('LOF Score')
ax2.set_ylabel('Density')
ax2.set_title('LOF Score Distribution with Thresholds')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_curve_and_threshold.png', dpi=300, bbox_inches='tight')
plt.show()

print("="*60)
print("ROC 曲線分析")
print("="*60)
print(f"AUC (Area Under Curve): {roc_auc:.3f}")
print(f"\n【解讀】")
print(f"  • AUC = {roc_auc:.3f} 表示隨機抽一筆異常與一筆正常，異常的 LOF 分數較高的機率為 {roc_auc*100:.1f}%")
print(f"  • ROC 曲線呈階梯狀，反映輕微異常與正常樣本的 LOF 分數重疊")

print(f"\n【閾值分析】contamination=0.083 對應的內部門檻為 LOF > {lof_threshold:.3f}")
print(f"{'Threshold':>10} {'Flagged':>8} {'TP':>4} {'FP':>4} {'Precision':>10} {'Recall':>8}")
for th in [1.2, 1.3, 1.5, 1.7, 2.0, 2.5]:
    flag = lof_scores > th
    tp = int(np.sum(flag & (y_true == -1)))
    fp = int(np.sum(flag & (y_true == 1)))
    prec = tp / flag.sum() if flag.sum() else 0.0
    print(f"{th:>10.1f} {flag.sum():>8d} {tp:>4d} {fp:>4d} {prec:>10.3f} {tp/np.sum(y_true == -1):>8.3f}")

print(f"\n【分級告警示例】")
print(f"  LOF < 1.5          → 正常操作（{np.sum(lof_scores < 1.5)} 個樣本）")
print(f"  1.5 ≤ LOF < 2.0    → 警告層級（{np.sum((lof_scores >= 1.5) & (lof_scores < 2.0))} 個樣本）")
print(f"  LOF ≥ 2.0          → 嚴重異常（{np.sum(lof_scores >= 2.0)} 個樣本）")
print(f"  閾值愈低 Recall 愈高、Precision 愈低；請依誤報與漏報的成本決定")
print("="*60)

ROC 曲線分析
AUC (Area Under Curve): 0.835

【解讀】
  • AUC = 0.835 表示隨機抽一筆異常與一筆正常，異常的 LOF 分數較高的機率為 83.5%
  • ROC 曲線呈階梯狀，反映輕微異常與正常樣本的 LOF 分數重疊

【閾值分析】contamination=0.083 對應的內部門檻為 LOF > 1.421
 Threshold  Flagged   TP   FP  Precision   Recall
       1.2       83   21   62      0.253    0.700
       1.3       53   18   35      0.340    0.600
       1.5       23   13   10      0.565    0.433
       1.7       11   10    1      0.909    0.333
       2.0        7    6    1      0.857    0.200
       2.5        6    6    0      1.000    0.200

【分級告警示例】
  LOF < 1.5          → 正常操作（337 個樣本）
  1.5 ≤ LOF < 2.0    → 警告層級（16 個樣本）
  LOF ≥ 2.0          → 嚴重異常（7 個樣本）
  閾值愈低 Recall 愈高、Precision 愈低；請依誤報與漏報的成本決定


### 4.3 LOF 分數詳細分析

In [11]:
# ========================================
# 4.3 LOF 分數詳細分析
# ========================================
# 找出 LOF 分數最高的前 10 個樣本
top10_anomaly_idx = np.argsort(lof_scores)[-10:][::-1]

print("="*70)
print("LOF 分數最高的前 10 個樣本（最可疑的異常）")
print("="*70)
print(f"{'Index':<8} {'Recipe':<10} {'Temp(C)':<10} {'Press(bar)':<12} "
      f"{'Flow(m3/h)':<12} {'LOF Score':<12} {'True Label':<12}")
print("-"*70)

for idx in top10_anomaly_idx:
    row = df.iloc[idx]
    true_label = 'Anomaly' if row['True_Label'] == -1 else 'Normal'
    print(f"{idx:<8} {row['Recipe']:<10} {row['Temperature']:<10.2f} "
          f"{row['Pressure']:<12.2f} {row['Flow_Rate']:<12.2f} "
          f"{row['LOF_Score']:<12.3f} {true_label:<12}")

# 統計各配方的 LOF 分數分布
print("\n" + "="*70)
print("各配方 LOF 分數統計")
print("="*70)
for recipe in ['A', 'B', 'C']:
    scores = df[df['Recipe'] == recipe]['LOF_Score'].values
    print(f"\n配方 {recipe}:")
    print(f"  數量: {len(scores)}")
    print(f"  平均: {scores.mean():.3f}")
    print(f"  標準差: {scores.std():.3f}")
    print(f"  最小值: {scores.min():.3f}")
    print(f"  最大值: {scores.max():.3f}")
    print(f"  超過 1.5 的數量: {np.sum(scores > 1.5)}")
    print(f"  超過 2.0 的數量: {np.sum(scores > 2.0)}")

# 異常樣本的 LOF 分數
anomaly_scores = df[df['Recipe'] == 'Anomaly']['LOF_Score'].values
print(f"\n異常樣本:")
print(f"  數量: {len(anomaly_scores)}")
print(f"  平均: {anomaly_scores.mean():.3f}")
print(f"  標準差: {anomaly_scores.std():.3f}")
print(f"  最小值: {anomaly_scores.min():.3f}")
print(f"  最大值: {anomaly_scores.max():.3f}")
print("="*70)

LOF 分數最高的前 10 個樣本（最可疑的異常）
Index    Recipe     Temp(C)    Press(bar)   Flow(m3/h)   LOF Score    True Label  
----------------------------------------------------------------------
359      Anomaly    88.00      2.50         3.00         4.282        Anomaly     
356      Anomaly    95.00      3.00         1.00         3.961        Anomaly     
358      Anomaly    40.00      4.50         2.50         3.760        Anomaly     
357      Anomaly    60.00      5.50         0.80         3.328        Anomaly     
355      Anomaly    55.00      2.00         0.50         2.966        Anomaly     
354      Anomaly    100.00     6.20         3.20         2.675        Anomaly     
69       A          87.06      5.21         3.46         2.090        Normal      
347      Anomaly    46.70      2.37         0.69         1.861        Anomaly     
346      Anomaly    82.53      5.29         1.59         1.805        Anomaly     
353      Anomaly    77.72      4.33         1.46         1.733        Ano

### 4.4 結果視覺化分析

In [12]:
# ========================================
# 4.4 LOF 結果視覺化
# ========================================
fig = plt.figure(figsize=(16, 12))

# 3D 散點圖 - LOF 檢測結果
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
normal_mask = df['LOF_Prediction'] == 'Normal'
anomaly_mask = df['LOF_Prediction'] == 'Anomaly'

ax1.scatter(df.loc[normal_mask, 'Temperature'], 
            df.loc[normal_mask, 'Pressure'],
            df.loc[normal_mask, 'Flow_Rate'],
            c='blue', label='Normal', alpha=0.6, s=30)
ax1.scatter(df.loc[anomaly_mask, 'Temperature'], 
            df.loc[anomaly_mask, 'Pressure'],
            df.loc[anomaly_mask, 'Flow_Rate'],
            c='red', label='Anomaly (LOF)', alpha=0.9, s=150, marker='x')

ax1.set_xlabel('Temperature (C)')
ax1.set_ylabel('Pressure (bar)')
ax1.set_zlabel('Flow Rate (m3/h)')
ax1.set_title('LOF Anomaly Detection Results (3D)')
ax1.legend()

# Temperature vs Pressure
ax2 = fig.add_subplot(2, 3, 2)
ax2.scatter(df.loc[normal_mask, 'Temperature'], 
            df.loc[normal_mask, 'Pressure'],
            c='blue', label='Normal', alpha=0.6, s=50)
ax2.scatter(df.loc[anomaly_mask, 'Temperature'], 
            df.loc[anomaly_mask, 'Pressure'],
            c='red', label='Anomaly (LOF)', alpha=0.9, s=150, marker='x')
ax2.set_xlabel('Temperature (C)')
ax2.set_ylabel('Pressure (bar)')
ax2.set_title('LOF Results: Temperature vs Pressure')
ax2.legend()
ax2.grid(True, alpha=0.3)

# LOF 分數分布
ax3 = fig.add_subplot(2, 3, 3)
hist_bins = np.linspace(lof_scores.min(), lof_scores.max(), 40)
ax3.hist(df[df['LOF_Prediction'] == 'Normal']['LOF_Score'], 
         bins=hist_bins, alpha=0.7, label='Predicted Normal', color='blue')
ax3.hist(df[df['LOF_Prediction'] == 'Anomaly']['LOF_Score'], 
         bins=hist_bins, alpha=0.7, label='Predicted Anomaly', color='red')
ax3.axvline(x=lof_threshold, color='green', linestyle='--', label=f'Decision threshold ({lof_threshold:.2f})')
ax3.set_xlabel('LOF Score')
ax3.set_ylabel('Frequency')
ax3.set_title('LOF Score Distribution')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 各配方的 LOF 分數箱型圖
ax4 = fig.add_subplot(2, 3, 4)
recipe_data = [df[df['Recipe'] == r]['LOF_Score'].values for r in ['A', 'B', 'C']]
ax4.boxplot(recipe_data)
ax4.set_xticklabels(['Recipe A', 'Recipe B', 'Recipe C'])
ax4.axhline(y=lof_threshold, color='green', linestyle='--', label=f'Decision threshold ({lof_threshold:.2f})')
ax4.set_ylabel('LOF Score')
ax4.set_title('LOF Scores by Recipe')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# 混淆矩陣熱力圖
ax5 = fig.add_subplot(2, 3, 5)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
ax5.set_xlabel('Predicted Label')
ax5.set_ylabel('True Label')
ax5.set_title('Confusion Matrix')

# LOF 分數排序圖（按真實標籤著色）
ax6 = fig.add_subplot(2, 3, 6)
order = np.argsort(lof_scores)
rank = np.arange(len(order))
sorted_scores = lof_scores[order]
sorted_true = y_true[order]
ax6.scatter(rank[sorted_true == 1], sorted_scores[sorted_true == 1],
            c='blue', label='True Normal', alpha=0.6, s=20)
ax6.scatter(rank[sorted_true == -1], sorted_scores[sorted_true == -1],
            c='red', label='True Anomaly', alpha=0.9, s=60, marker='x')
ax6.axhline(y=lof_threshold, color='green', linestyle='--', label='Decision threshold')
ax6.set_xlabel('Rank (sorted by LOF score)')
ax6.set_ylabel('LOF Score')
ax6.set_title('Sorted LOF Scores (by True Label)')
ax6.legend()
ax6.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.subplots_adjust(wspace=0.4)
plt.savefig(FIG_DIR / 'lof_results_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 結果視覺化完成")

✓ 結果視覺化完成


### 4.5 異常嚴重程度與配方別分析

In [13]:
# ========================================
# 4.5 依異常嚴重程度與配方分析檢測結果
# ========================================
is_anom = df['True_Label'] == -1
flagged = df['LOF_Prediction'] == 'Anomaly'

sev_table = (df[is_anom].assign(Detected=flagged[is_anom])
             .groupby('Anomaly_Severity')
             .agg(n=('Detected', 'size'), detected=('Detected', 'sum'),
                  mean_LOF=('LOF_Score', 'mean'), min_LOF=('LOF_Score', 'min'), max_LOF=('LOF_Score', 'max'))
             .reindex(['Mild', 'Moderate', 'Severe']))
sev_table['recall'] = sev_table['detected'] / sev_table['n']

fp_table = (df[~is_anom].assign(FalseAlarm=flagged[~is_anom])
            .groupby('Recipe')
            .agg(n=('FalseAlarm', 'size'), false_alarms=('FalseAlarm', 'sum')))
fp_table['false_alarm_rate'] = fp_table['false_alarms'] / fp_table['n']

print("依異常嚴重程度的檢出結果:")
print(sev_table.round(3).to_string())
print("\n正常批次依配方的誤報結果:")
print(fp_table.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(sev_table.index, sev_table['recall'], color=['gold', 'orange', 'red'], edgecolor='black')
for i in range(len(sev_table)):
    axes[0].text(i, sev_table['recall'].iloc[i] + 0.02,
                 f"{int(sev_table['detected'].iloc[i])}/{int(sev_table['n'].iloc[i])}", ha='center')
axes[0].set_ylim(0, 1.1)
axes[0].set_xlabel('Anomaly Severity')
axes[0].set_ylabel('Recall')
axes[0].set_title('Detection Rate by Anomaly Severity')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(fp_table.index, fp_table['false_alarm_rate'], color=['tab:blue', 'tab:green', 'tab:orange'], edgecolor='black')
for i in range(len(fp_table)):
    axes[1].text(i, fp_table['false_alarm_rate'].iloc[i] + 0.002,
                 f"{int(fp_table['false_alarms'].iloc[i])}/{int(fp_table['n'].iloc[i])}", ha='center')
axes[1].set_ylim(0, fp_table['false_alarm_rate'].max() * 1.2)
axes[1].set_xlabel('Recipe')
axes[1].set_ylabel('False Alarm Rate')
axes[1].set_title('False Alarm Rate by Recipe (Normal Batches)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIG_DIR / 'detection_by_severity_recipe.png', dpi=300, bbox_inches='tight')
plt.show()

依異常嚴重程度的檢出結果:
                   n  detected  mean_LOF  min_LOF  max_LOF  recall
Anomaly_Severity                                                  
Mild              12         2     1.193    0.964    1.501   0.167
Moderate          12         7     1.494    1.012    1.861   0.583
Severe             6         6     3.495    2.675    4.282   1.000

正常批次依配方的誤報結果:
          n  false_alarms  false_alarm_rate
Recipe                                     
A       150             8             0.053
B       100             5             0.050
C        80             2             0.025


---
## 5. 新樣本預測（Novelty Detection 模式）

實際應用中，需要對「新的批次數據」判斷是否異常，此時使用 LOF 的 `novelty=True` 模式：以歷史正常數據擬合，之後呼叫 `predict()` / `score_samples()` 評估新樣本。

### 5.1 建立 Novelty Detection 模式的 LOF 模型

In [14]:
# ========================================
# 5.1 建立 Novelty Detection 模式的 LOF 模型
# ========================================
# 部署情境：scaler 與模型都只用「正常」歷史數據擬合，不可混入異常批次
X_normal_only = df.loc[df['True_Label'] == 1, feature_names].values
scaler_nov = StandardScaler().fit(X_normal_only)
X_normal_scaled = scaler_nov.transform(X_normal_only)

lof_novelty = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,   # 訓練數據為純正常批次，此值約為可接受的誤報率
    novelty=True,
    metric='euclidean'
)
lof_novelty.fit(X_normal_scaled)

print("="*60)
print("Novelty Detection 模式的 LOF 模型訓練完成")
print("="*60)
print(f"訓練數據數量: {len(X_normal_scaled)}")
print(f"特徵數量: {X_normal_scaled.shape[1]}")
print(f"n_neighbors: {lof_novelty.n_neighbors}")
print(f"contamination: {lof_novelty.contamination}")
print(f"判定門檻: LOF > {-lof_novelty.offset_:.3f}")
print("="*60)

Novelty Detection 模式的 LOF 模型訓練完成
訓練數據數量: 330
特徵數量: 3
n_neighbors: 20
contamination: 0.05
判定門檻: LOF > 1.438


### 5.2 模擬新批次數據進行預測

新批次包含 60 筆正常（三種配方各 20 筆）與 36 筆異常（輕微、中度、嚴重各 12 筆）。異常樣本由隨機挑選的 2 個特徵偏移 $1.5\sigma$、$2.5\sigma$、$4\sigma$ 生成。

In [15]:
# ========================================
# 5.2 模擬新批次數據進行預測
# ========================================
rng = np.random.default_rng(SEED + 1)
recipe_params = {'A': ([85, 5.0, 2.5], [4, 0.4, 0.25]),
                 'B': ([65, 3.5, 1.8], [4, 0.4, 0.2]),
                 'C': ([45, 2.5, 1.2], [3, 0.3, 0.15])}
shift_sigma = {'Mild': 1.5, 'Moderate': 2.5, 'Severe': 4.0}

new_normal = np.vstack([rng.normal(mu, sd, size=(20, 3)) for mu, sd in recipe_params.values()])

new_anomaly, new_severity = [], []
for mu, sd in recipe_params.values():
    mu, sd = np.array(mu), np.array(sd)
    for sev, d in shift_sigma.items():
        for _ in range(4):
            x = rng.normal(mu, sd)
            j = rng.choice(3, size=2, replace=False)      # 隨機挑 2 個特徵偏移
            x[j] += rng.choice([-1, 1], size=2) * d * sd[j]
            new_anomaly.append(x)
            new_severity.append(sev)

X_new = np.vstack([new_normal, np.array(new_anomaly)])
X_new_scaled = scaler_nov.transform(X_new)

y_new_true = np.array([1] * len(new_normal) + [-1] * len(new_anomaly))
y_new_pred = lof_novelty.predict(X_new_scaled)
new_scores = -lof_novelty.score_samples(X_new_scaled)

df_new = pd.DataFrame(X_new, columns=feature_names)
df_new['Prediction'] = ['Normal' if p == 1 else 'Anomaly' for p in y_new_pred]
df_new['LOF_Score'] = new_scores
df_new['Sample_Type'] = ['Normal'] * len(new_normal) + ['Anomaly'] * len(new_anomaly)
df_new['Severity'] = ['Normal'] * len(new_normal) + new_severity

cm_new = confusion_matrix(y_new_true, y_new_pred, labels=[1, -1])
precision_new = precision_score(y_new_true, y_new_pred, pos_label=-1)
recall_new = recall_score(y_new_true, y_new_pred, pos_label=-1)
f1_new = f1_score(y_new_true, y_new_pred, pos_label=-1)
auc_new = roc_auc_score(y_new_true == -1, new_scores)

print("="*70)
print("新批次數據異常檢測結果")
print("="*70)
print(f"{'':>15} {'Predicted Normal':>18} {'Predicted Anomaly':>18}")
print(f"{'True Normal':>15} {cm_new[0, 0]:>18} {cm_new[0, 1]:>18}")
print(f"{'True Anomaly':>15} {cm_new[1, 0]:>18} {cm_new[1, 1]:>18}")
print(f"\nPrecision: {precision_new:.3f} | Recall: {recall_new:.3f} | F1-Score: {f1_new:.3f} | AUC: {auc_new:.3f}")

print("\n依嚴重程度的檢出率:")
for sev in shift_sigma:
    m = (df_new['Severity'] == sev).values
    print(f"  {sev:<9} (偏移 {shift_sigma[sev]}σ): {np.sum(y_new_pred[m] == -1)}/{m.sum()}")

print("\ncontamination（判定門檻）對結果的影響:")
print(f"{'contamination':>14} {'threshold':>10} {'FP':>4} {'FN':>4} {'Precision':>10} {'Recall':>8}")
for c in [0.01, 0.05, 0.10]:
    m_c = LocalOutlierFactor(n_neighbors=20, contamination=c, novelty=True).fit(X_normal_scaled)
    p_c = m_c.predict(X_new_scaled)
    fp = int(np.sum((p_c == -1) & (y_new_true == 1)))
    fn = int(np.sum((p_c == 1) & (y_new_true == -1)))
    tp = int(np.sum((p_c == -1) & (y_new_true == -1)))
    print(f"{c:>14.2f} {-m_c.offset_:>10.3f} {fp:>4d} {fn:>4d} {tp/max(tp+fp, 1):>10.3f} {tp/np.sum(y_new_true == -1):>8.3f}")
print("="*70)

新批次數據異常檢測結果
                  Predicted Normal  Predicted Anomaly
    True Normal                 54                  6
   True Anomaly                 13                 23

Precision: 0.793 | Recall: 0.639 | F1-Score: 0.708 | AUC: 0.924

依嚴重程度的檢出率:
  Mild      (偏移 1.5σ): 3/12
  Moderate  (偏移 2.5σ): 9/12
  Severe    (偏移 4.0σ): 11/12

contamination（判定門檻）對結果的影響:
 contamination  threshold   FP   FN  Precision   Recall
          0.01      1.757    0   19      1.000    0.472
          0.05      1.438    6   13      0.793    0.639
          0.10      1.333    8    7      0.784    0.806


### 5.3 視覺化新樣本預測結果

In [16]:
# ========================================
# 5.3 視覺化新樣本預測結果
# ========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.scatter(df.loc[df['True_Label'] == 1, 'Temperature'],
            df.loc[df['True_Label'] == 1, 'Pressure'],
            c='lightgray', alpha=0.5, s=20, label='Training (normal)')
nm = df_new['Sample_Type'] == 'Normal'
ax1.scatter(df_new.loc[nm, 'Temperature'], df_new.loc[nm, 'Pressure'],
            c='blue', marker='o', s=50, edgecolors='black', linewidths=0.8,
            label='New Normal', zorder=5)
ax1.scatter(df_new.loc[~nm, 'Temperature'], df_new.loc[~nm, 'Pressure'],
            c='red', marker='x', s=80, linewidths=2, label='New Anomaly', zorder=5)
ax1.set_xlabel('Temperature (C)')
ax1.set_ylabel('Pressure (bar)')
ax1.set_title('New Samples in Process Space')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
bins = np.linspace(0.8, max(new_scores.max(), 3.0), 30)
ax2.hist(df_new.loc[nm, 'LOF_Score'], bins=bins, alpha=0.7, color='blue', label='New Normal')
ax2.hist(df_new.loc[~nm, 'LOF_Score'], bins=bins, alpha=0.7, color='red', label='New Anomaly')
ax2.axvline(x=-lof_novelty.offset_, color='green', linestyle='--', linewidth=2,
            label=f'Decision threshold ({-lof_novelty.offset_:.2f})')
ax2.set_xlabel('LOF Score')
ax2.set_ylabel('Count')
ax2.set_title('LOF Scores of New Samples')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIG_DIR / 'novelty_detection_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 新樣本預測結果視覺化完成")

✓ 新樣本預測結果視覺化完成


### 5.4 兩種 LOF 模式比較

| 特性 | `novelty=False` | `novelty=True` |
|------|-----------------|----------------|
| 使用場景 | 對手上這批數據做離線異常檢測 | 對新樣本做線上判定 |
| 訓練數據 | 可含異常（異常也被評分） | 應僅含正常數據 |
| 可用方法 | `fit_predict()`、`negative_outlier_factor_` | `fit()` 後 `predict()`、`score_samples()` |
| 可否預測新樣本 | 否 | 是 |

**注意**：在 `novelty=True` 模式下，不應對訓練數據本身呼叫 `predict()` 評估效能，因為每個訓練點會把自己算進鄰域。評估應使用獨立的新樣本。

**實務流程**：

1. 離線階段：以歷史正常批次擬合 `StandardScaler` 與 `novelty=True` 的 LOF，儲存兩者。
2. 線上階段：載入 scaler 與模型，新批次先標準化，再用 `score_samples()` 取得 LOF 分數、`predict()` 取得判定。
3. 告警：以 LOF 分數設定分級門檻（如 1.5 警告、2.0 嚴重），門檻須依誤報與漏報成本與歷史數據校準。

### 5.5 儲存 Novelty 模型與 Scaler

In [17]:
# ========================================
# 5.5 儲存 Novelty 模型與 Scaler，並重新載入驗證
# ========================================
scaler_path = MODEL_DIR / 'lof_novelty_scaler.pkl'
model_path = MODEL_DIR / 'lof_novelty_model.pkl'
meta_path = MODEL_DIR / 'lof_novelty_meta.json'

joblib.dump(scaler_nov, scaler_path)
joblib.dump(lof_novelty, model_path)

meta = {
    'n_neighbors': 20,
    'contamination': 0.05,
    'features': feature_names,
    'n_training_samples': int(len(X_normal_scaled)),
    'decision_threshold_LOF': float(-lof_novelty.offset_),
    'new_sample_test': {'precision': float(precision_new), 'recall': float(recall_new),
                        'f1': float(f1_new), 'auc': float(auc_new)},
}
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

# 重新載入後的預測必須與原模型一致
scaler_loaded = joblib.load(scaler_path)
model_loaded = joblib.load(model_path)
assert np.array_equal(model_loaded.predict(scaler_loaded.transform(X_new)), y_new_pred)

print(f"✓ Scaler 已儲存: {scaler_path.name}")
print(f"✓ 模型已儲存: {model_path.name}")
print(f"✓ 參數與效能摘要已儲存: {meta_path.name}")
print("✓ 重新載入驗證通過（預測結果與原模型一致）")

✓ Scaler 已儲存: lof_novelty_scaler.pkl
✓ 模型已儲存: lof_novelty_model.pkl
✓ 參數與效能摘要已儲存: lof_novelty_meta.json
✓ 重新載入驗證通過（預測結果與原模型一致）


---
## 6. 與 Isolation Forest 比較

### 6.1 訓練 Isolation Forest 模型

In [18]:
# ========================================
# 6.1 建立並訓練 Isolation Forest 模型
# ========================================
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.083,
    random_state=SEED
)

y_pred_iso = iso_forest.fit_predict(X_scaled)
iso_scores = -iso_forest.score_samples(X_scaled)

df['ISO_Prediction'] = ['Normal' if p == 1 else 'Anomaly' for p in y_pred_iso]
df['ISO_Score'] = iso_scores

cm_iso = confusion_matrix(y_true, y_pred_iso, labels=[1, -1])
precision_iso = precision_score(y_true, y_pred_iso, pos_label=-1)
recall_iso = recall_score(y_true, y_pred_iso, pos_label=-1)
f1_iso = f1_score(y_true, y_pred_iso, pos_label=-1)
auc_iso = roc_auc_score(y_true == -1, iso_scores)

print("="*60)
print("Isolation Forest 模型效能評估")
print("="*60)
print(f"檢測到的異常批次: {np.sum(y_pred_iso == -1)}")
print(f"  Precision (精確率): {precision_iso:.3f}")
print(f"  Recall (召回率):    {recall_iso:.3f}")
print(f"  F1-Score:           {f1_iso:.3f}")
print(f"  AUC:                {auc_iso:.3f}")

Isolation Forest 模型效能評估
檢測到的異常批次: 30
  Precision (精確率): 0.400
  Recall (召回率):    0.400
  F1-Score:           0.400
  AUC:                0.777


### 6.2 LOF vs Isolation Forest 比較（單一數據集，seed = 42）

In [19]:
# ========================================
# 6.2 LOF vs Isolation Forest 比較
# ========================================
# 建立比較表格
comparison = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-Score', 'AUC'],
    'LOF': [precision, recall, f1, roc_auc],
    'Isolation Forest': [precision_iso, recall_iso, f1_iso, auc_iso]
})

print("="*60)
print("LOF vs Isolation Forest 效能比較")
print("="*60)
print(comparison.round(3).to_string(index=False))

# 視覺化比較
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 效能指標比較
metrics = ['Precision', 'Recall', 'F1-Score', 'AUC']
lof_metrics = [precision, recall, f1, roc_auc]
iso_metrics = [precision_iso, recall_iso, f1_iso, auc_iso]

x = np.arange(len(metrics))
width = 0.35

axes[0, 0].bar(x - width/2, lof_metrics, width, label='LOF', alpha=0.8)
axes[0, 0].bar(x + width/2, iso_metrics, width, label='Isolation Forest', alpha=0.8)
axes[0, 0].set_xlabel('Metric')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Performance Metrics Comparison')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 異常檢測結果比較（溫度 vs 壓力）
axes[0, 1].scatter(df.loc[df['LOF_Prediction'] == 'Normal', 'Temperature'], 
                   df.loc[df['LOF_Prediction'] == 'Normal', 'Pressure'],
                   c='blue', label='Normal', alpha=0.5, s=30)
axes[0, 1].scatter(df.loc[df['LOF_Prediction'] == 'Anomaly', 'Temperature'], 
                   df.loc[df['LOF_Prediction'] == 'Anomaly', 'Pressure'],
                   c='red', label='LOF Anomaly', alpha=0.8, s=100, marker='x')
axes[0, 1].set_xlabel('Temperature (C)')
axes[0, 1].set_ylabel('Pressure (bar)')
axes[0, 1].set_title('LOF Detection Results')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].scatter(df.loc[df['ISO_Prediction'] == 'Normal', 'Temperature'], 
                   df.loc[df['ISO_Prediction'] == 'Normal', 'Pressure'],
                   c='blue', label='Normal', alpha=0.5, s=30)
axes[1, 0].scatter(df.loc[df['ISO_Prediction'] == 'Anomaly', 'Temperature'], 
                   df.loc[df['ISO_Prediction'] == 'Anomaly', 'Pressure'],
                   c='orange', label='ISO Anomaly', alpha=0.8, s=100, marker='x')
axes[1, 0].set_xlabel('Temperature (C)')
axes[1, 0].set_ylabel('Pressure (bar)')
axes[1, 0].set_title('Isolation Forest Detection Results')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 檢測一致性分析
both_anomaly = (df['LOF_Prediction'] == 'Anomaly') & (df['ISO_Prediction'] == 'Anomaly')
only_lof = (df['LOF_Prediction'] == 'Anomaly') & (df['ISO_Prediction'] == 'Normal')
only_iso = (df['LOF_Prediction'] == 'Normal') & (df['ISO_Prediction'] == 'Anomaly')
both_normal = (df['LOF_Prediction'] == 'Normal') & (df['ISO_Prediction'] == 'Normal')

agreement_counts = [
    np.sum(both_anomaly),
    np.sum(only_lof),
    np.sum(only_iso),
    np.sum(both_normal)
]
labels = ['Both Anomaly', 'Only LOF', 'Only ISO', 'Both Normal']

axes[1, 1].bar(labels, agreement_counts, alpha=0.8, color=['red', 'blue', 'orange', 'green'])
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Detection Agreement Analysis')
axes[1, 1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(FIG_DIR / 'lof_vs_isoforest_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n檢測一致性分析:")
print(f"  兩者都判定為異常: {np.sum(both_anomaly)}")
print(f"  僅 LOF 判定為異常: {np.sum(only_lof)}")
print(f"  僅 ISO 判定為異常: {np.sum(only_iso)}")
print(f"  兩者都判定為正常: {np.sum(both_normal)}")

LOF vs Isolation Forest 效能比較
   Metric   LOF  Isolation Forest
Precision 0.500             0.400
   Recall 0.500             0.400
 F1-Score 0.500             0.400
      AUC 0.835             0.777



檢測一致性分析:
  兩者都判定為異常: 22
  僅 LOF 判定為異常: 8
  僅 ISO 判定為異常: 8
  兩者都判定為正常: 322


### 6.3 多組隨機種子的穩健性比較（含全域 kNN 距離基準線）

單一數據集（30 筆異常）的 F1 只差幾筆樣本，不足以判斷哪個方法較好。這裡以 20 組不同隨機種子重複生成數據，並加入「全域 kNN 距離」（第 $k$ 近鄰距離，不做局部密度比較）作為基準線，檢驗 LOF 的局部密度比較是否帶來額外價值。

In [20]:
# ========================================
# 6.3 多組隨機種子：LOF vs Isolation Forest vs 全域 kNN 距離
# ========================================
def knn_distance_score(X, k=20):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    dist, _ = nn.kneighbors(X)
    return dist[:, -1]          # 到第 k 近鄰的距離（全域指標）


records = []
for s in range(42, 62):
    d = generate_batch_reactor_data(seed=s)
    Xs = StandardScaler().fit_transform(d[feature_names].values)
    y_bin = (d['True_Label'].values == -1).astype(int)

    lof_s = LocalOutlierFactor(n_neighbors=20, contamination=0.083)
    pred_lof_s = (lof_s.fit_predict(Xs) == -1).astype(int)
    score_lof_s = -lof_s.negative_outlier_factor_

    iso_s = IsolationForest(n_estimators=100, contamination=0.083, random_state=SEED).fit(Xs)
    pred_iso_s = (iso_s.predict(Xs) == -1).astype(int)
    score_iso_s = -iso_s.score_samples(Xs)

    records.append({
        'seed': s,
        'AUC_LOF': roc_auc_score(y_bin, score_lof_s),
        'AUC_ISO': roc_auc_score(y_bin, score_iso_s),
        'AUC_kNN': roc_auc_score(y_bin, knn_distance_score(Xs, 20)),
        'F1_LOF': f1_score(y_bin, pred_lof_s),
        'F1_ISO': f1_score(y_bin, pred_iso_s),
    })
np.random.seed(SEED)            # generate_batch_reactor_data 會重設全域種子，這裡還原

ms_df = pd.DataFrame(records)
ms_summary = ms_df.drop(columns='seed').agg(['mean', 'std', 'min', 'max']).T
print("20 組隨機種子（42~61）的結果:")
print(ms_summary.round(3).to_string())
print(f"\nAUC：LOF 高於 Isolation Forest 的種子數: {(ms_df['AUC_LOF'] > ms_df['AUC_ISO']).sum()}/20")
print(f"AUC：LOF 高於 全域 kNN 距離 的種子數:   {(ms_df['AUC_LOF'] > ms_df['AUC_kNN']).sum()}/20")
print(f"F1 ：LOF 高於 Isolation Forest 的種子數: {(ms_df['F1_LOF'] > ms_df['F1_ISO']).sum()}/20")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].boxplot([ms_df['AUC_LOF'], ms_df['AUC_ISO'], ms_df['AUC_kNN']])
axes[0].set_xticklabels(['LOF', 'Isolation Forest', 'Global kNN distance'])
axes[0].set_ylabel('AUC')
axes[0].set_title('AUC over 20 Random Seeds')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].boxplot([ms_df['F1_LOF'], ms_df['F1_ISO']])
axes[1].set_xticklabels(['LOF', 'Isolation Forest'])
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1-Score over 20 Random Seeds')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIG_DIR / 'multi_seed_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

20 組隨機種子（42~61）的結果:
          mean    std    min    max
AUC_LOF  0.817  0.037  0.738  0.876
AUC_ISO  0.784  0.026  0.732  0.831
AUC_kNN  0.819  0.041  0.735  0.887
F1_LOF   0.422  0.064  0.333  0.567
F1_ISO   0.360  0.035  0.300  0.433

AUC：LOF 高於 Isolation Forest 的種子數: 17/20
AUC：LOF 高於 全域 kNN 距離 的種子數:   7/20
F1 ：LOF 高於 Isolation Forest 的種子數: 15/20


### 6.4 多密度玩具範例：LOF 的局部性優勢

批次反應器數據經標準化後，三種配方的密度差異有限，所以全域 kNN 距離的表現與 LOF 接近。以下用密度差異極大的 2D 數據說明 LOF 的局部性：一個緻密群集（200 點，$\sigma = 0.3$）、一個稀疏群集（100 點，$\sigma = 2$），以及 8 個緊貼緻密群集外圍的局部異常點。

In [21]:
# ========================================
# 6.4 多密度玩具範例
# ========================================
toy_auc = []
for s in range(10):
    r_toy = np.random.default_rng(s)
    dense = r_toy.normal([0, 0], 0.3, size=(200, 2))
    sparse = r_toy.normal([8, 8], 2.0, size=(100, 2))
    ang = r_toy.uniform(0, 2 * np.pi, 8)
    local_out = np.c_[1.8 * np.cos(ang), 1.8 * np.sin(ang)] + r_toy.normal(0, 0.1, (8, 2))
    X_toy = np.vstack([dense, sparse, local_out])
    y_toy = np.r_[np.zeros(300), np.ones(8)]

    s_lof_toy = -LocalOutlierFactor(n_neighbors=20).fit(X_toy).negative_outlier_factor_
    s_knn_toy = knn_distance_score(X_toy, 20)
    s_iso_toy = -IsolationForest(random_state=SEED).fit(X_toy).score_samples(X_toy)
    toy_auc.append([roc_auc_score(y_toy, s_lof_toy), roc_auc_score(y_toy, s_knn_toy), roc_auc_score(y_toy, s_iso_toy)])
    if s == 0:
        X_toy0, y_toy0, s_lof_toy0 = X_toy, y_toy, s_lof_toy

toy_auc = np.array(toy_auc)
toy_mean, toy_std = toy_auc.mean(axis=0), toy_auc.std(axis=0)
print("10 組隨機種子的 AUC（平均 ± 標準差）:")
for name, m, sd in zip(['LOF', 'Global kNN distance', 'Isolation Forest'], toy_mean, toy_std):
    print(f"  {name:<20}: {m:.3f} ± {sd:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax1 = axes[0]
ax1.scatter(X_toy0[y_toy0 == 0, 0], X_toy0[y_toy0 == 0, 1], c='lightgray', s=20, label='Normal')
sc = ax1.scatter(X_toy0[y_toy0 == 1, 0], X_toy0[y_toy0 == 1, 1], c=s_lof_toy0[y_toy0 == 1],
                 cmap='Reds', vmin=0, s=90, marker='X', edgecolors='black', label='Local outlier')
plt.colorbar(sc, ax=ax1, label='LOF Score')
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Dense + Sparse Clusters with Local Outliers')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.bar(['LOF', 'Global kNN\ndistance', 'Isolation\nForest'], toy_mean, yerr=toy_std, capsize=5,
        color=['tab:red', 'tab:gray', 'tab:orange'], edgecolor='black')
ax2.set_ylim(0.5, 1.05)
ax2.set_ylabel('AUC')
ax2.set_title('AUC over 10 Random Seeds')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIG_DIR / 'multidensity_toy.png', dpi=300, bbox_inches='tight')
plt.show()

10 組隨機種子的 AUC（平均 ± 標準差）:
  LOF                 : 1.000 ± 0.001
  Global kNN distance : 0.737 ± 0.027
  Isolation Forest    : 0.973 ± 0.009


---
## 7. 總結與實務建議

### 7.1 實驗結果總結

In [22]:
print("="*70)
print("實驗結果總結")
print("="*70)

print("\n1. 內部異常檢測（novelty=False, k=20, contamination=0.083, seed=42）")
print(f"   Precision = {precision:.3f}, Recall = {recall:.3f}, F1 = {f1:.3f}, AUC = {roc_auc:.3f}")
print("   Precision 與 Recall 相同，因為預測異常數 (30) 等於真實異常數 (30)")
for sev in sev_table.index:
    print(f"   {sev:<9}異常檢出: {int(sev_table.loc[sev, 'detected'])}/{int(sev_table.loc[sev, 'n'])}")
print("   正常批次誤報: " + ", ".join(f"配方 {r}: {int(fp_table.loc[r, 'false_alarms'])}/{int(fp_table.loc[r, 'n'])}" for r in fp_table.index))

print("\n2. 超參數 k 的影響")
print(f"   AUC 最高的 k = {best_k_auc}，F1 最高的 k = {best_k_f1}；k 過小 (5) AUC 明顯偏低，k 在 20~50 之間 AUC 變化不大")
print("   以真實標籤選 k 僅供教學；實務請用獨立驗證集或檢查鄰近 k 的穩定性")

print("\n3. 新樣本預測（novelty=True，60 筆正常 + 36 筆異常）")
print(f"   Precision = {precision_new:.3f}, Recall = {recall_new:.3f}, F1 = {f1_new:.3f}, AUC = {auc_new:.3f}")
print("   此測試集的異常偏移量與難度由人為設定，數值不可與第 1 項直接比較")

print("\n4. LOF vs Isolation Forest（20 組隨機種子）")
print(f"   AUC：LOF {ms_summary.loc['AUC_LOF', 'mean']:.3f} ± {ms_summary.loc['AUC_LOF', 'std']:.3f}，"
      f"Isolation Forest {ms_summary.loc['AUC_ISO', 'mean']:.3f} ± {ms_summary.loc['AUC_ISO', 'std']:.3f}，"
      f"全域 kNN 距離 {ms_summary.loc['AUC_kNN', 'mean']:.3f} ± {ms_summary.loc['AUC_kNN', 'std']:.3f}")
print(f"   F1 ：LOF {ms_summary.loc['F1_LOF', 'mean']:.3f}，Isolation Forest {ms_summary.loc['F1_ISO', 'mean']:.3f}")
print("   本數據標準化後各配方密度差異有限，LOF 與全域 kNN 距離表現相近")
print(f"   在密度差異極大的玩具範例中，AUC：LOF {toy_mean[0]:.3f}、全域 kNN 距離 {toy_mean[1]:.3f}、Isolation Forest {toy_mean[2]:.3f}")

print("\n5. 實務應用建議")
print("   ✓ 先標準化再計算距離")
print("   ✓ novelty=True 模式的 scaler 與模型只用正常數據擬合")
print("   ✓ 以 LOF 分數設定分級門檻，並依誤報/漏報成本與歷史數據校準")
print("   ✓ 輕微異常與正常數據重疊時，Recall 有上限，需搭配領域知識與其他監控指標")
print("   ✓ 定期以新的正常批次重新擬合模型")

print("\n6. LOF 的限制")
print("   ✗ 暴力搜尋為 O(n²)；低維數據搭配 KD-tree/Ball-tree 可顯著加快，但高維時效益下降")
print("   ✗ 高維數據的距離失去區分力，建議先降維或做特徵選擇")
print("   ✗ novelty=False 模式無法預測新樣本")
print("   ✗ 對 k 與特徵標度敏感")
print("="*70)

實驗結果總結

1. 內部異常檢測（novelty=False, k=20, contamination=0.083, seed=42）
   Precision = 0.500, Recall = 0.500, F1 = 0.500, AUC = 0.835
   Precision 與 Recall 相同，因為預測異常數 (30) 等於真實異常數 (30)
   Mild     異常檢出: 2/12
   Moderate 異常檢出: 7/12
   Severe   異常檢出: 6/6
   正常批次誤報: 配方 A: 8/150, 配方 B: 5/100, 配方 C: 2/80

2. 超參數 k 的影響
   AUC 最高的 k = 50，F1 最高的 k = 15；k 過小 (5) AUC 明顯偏低，k 在 20~50 之間 AUC 變化不大
   以真實標籤選 k 僅供教學；實務請用獨立驗證集或檢查鄰近 k 的穩定性

3. 新樣本預測（novelty=True，60 筆正常 + 36 筆異常）
   Precision = 0.793, Recall = 0.639, F1 = 0.708, AUC = 0.924
   此測試集的異常偏移量與難度由人為設定，數值不可與第 1 項直接比較

4. LOF vs Isolation Forest（20 組隨機種子）
   AUC：LOF 0.817 ± 0.037，Isolation Forest 0.784 ± 0.026，全域 kNN 距離 0.819 ± 0.041
   F1 ：LOF 0.422，Isolation Forest 0.360
   本數據標準化後各配方密度差異有限，LOF 與全域 kNN 距離表現相近
   在密度差異極大的玩具範例中，AUC：LOF 1.000、全域 kNN 距離 0.737、Isolation Forest 0.973

5. 實務應用建議
   ✓ 先標準化再計算距離
   ✓ novelty=True 模式的 scaler 與模型只用正常數據擬合
   ✓ 以 LOF 分數設定分級門檻，並依誤報/漏報成本與歷史數據校準
   ✓ 輕微異常與正常數據重疊時，Recall 有上限，需搭配領域知識與其他監控指標
   ✓ 定期以新的正常批次重新擬合模型

6